In [6]:
import tensorflow as tf
import numpy as np 
from tensorflow import keras 

latent_dim=100

def build_generator():
    model = keras.Sequential([
        keras.layers.Dense(256, input_dim=latent_dim),
        keras.layers.LeakyReLU(0.2),
        keras.layers.Dense(512),
        keras.layers.LeakyReLU(0.2),
        keras.layers.Dense(784, activation='tanh'),
    ])

def build_discriminator():
    model = keras.Sequential([
        keras.layers.Dense(512, input_dim=784),
        keras.layers.LeakyReLU(0.2),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(256),
        keras.layers.LeakyReLU(0.2),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(1, activation='sigmoid'),
    ])

generator = build_generator()
discriminator = build_discriminator()


gen_optim = keras.optimizers.Adam(0.0002, beta_1=0.5)
disc_optim = keras.optimizers.Adam(0.0002, beta_1=0.5)

bce = tf.keras.losses.BinaryCrossentropy

@tf.function
def train_step(real_images):
    batch_size = tf.shape(real_images)[0]

    noise = tf.random.normal([latent_dim, batch_size])

    with tf.GradientTape as disc_tape:
        fake_images = generator(noise, trainable=True)

        real_output = discriminator(real_images, trainable=True)
        fake_output = discriminator(fake_images, trainable=True)

        real_loss = bce(tf.ones_like(real_output), real_output)
        fake_loss = bce(tf.zeros_like(fake_output), fake_output)

        disc_loss = real_loss+ fake_loss

    disc_gradient = disc_tape.gradient(disc_loss, discriminator.learnable_variables)
    disc_optim.apply_gradients(zip(disc_gradient, discriminator.learnable_variables))

    noise = tf.random.normal([latent_dim, batch_size])

    with tf.GradientTape as gen_tape:
        fake_images = generator(noise, trainable=True)

        fake_output = discriminator(fake_images, trainable=True)

        gen_loss = bce(tf.ones_like(fake_output), fake_output)


    gen_gradient = gen_tape.gradient(disc_loss, generator.learnable_variables)
    gen_optim.apply_gradients(zip(gen_gradient, generator.learnable_variables))

    return disc_loss, gen_loss

def train(dataset, epochs):
    for epoch in epochs:
        for batch in dataset:
            disc_loss, gen_loss = train_step(batch)

        print(f'epoch {epoch}, gen_loss: {gen_loss}, Disc_loss: {disc_loss}')

(x_train,_), _ = tf.keras.datasets.mnist.load_data()
x_train = tf.reshape(28, 28, 32)(x_train)
x_train = x_train/256

train(x_train, 10)



2025-12-02 09:55:14.923920: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: INVALID_ARGUMENT: Input to reshape is a tensor with 1 values, but the requested shape has 28


InvalidArgumentError: {{function_node __wrapped__Reshape_device_/job:localhost/replica:0/task:0/device:CPU:0}} Input to reshape is a tensor with 1 values, but the requested shape has 28 [Op:Reshape] name: 32